# Sentiment, Market Merge, and Connectedness Analysis

## Project context

This notebook uses the recovered prototype data files for the News Sentiment and Market Connectedness Engine. The goal is to clean sentiment logs, inspect scraped news, merge sentiment with available market fields, summarize simple signal labels, and run a transparent connectedness fallback when the data is too small for formal GFEVD.

## Phase 1 objective

- Load copied raw prototype files.
- Check whether legacy scripts are real or placeholders.
- Standardize sentiment fields.
- Standardize market columns in the merged JSON data.
- Create clean sentiment, signal, merged, and connectedness outputs.
- Avoid trading performance claims and document limitations.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.config import RAW_DATA_DIR, OUTPUTS_DIR, FIGURES_DIR, LEGACY_DIR, REPORTS_DIR
from src.ingestion import load_csv, load_json_records, load_available_raw_datasets, inspect_dataframe, save_dataframe
from src.sentiment import (
    standardize_sentiment_log,
    standardize_scraped_news,
    summarize_sentiment_by_date,
    summarize_sentiment_by_company,
    create_signal_summary,
    score_sentiment_dataframe,
)
from src.market_data import standardize_market_columns, summarize_market_data, calculate_market_returns, identify_market_columns
from src.merge import merge_sentiment_market, inspect_merged_dataset, identify_join_keys, create_modeling_dataset
from src.connectedness import (
    prepare_connectedness_inputs,
    calculate_correlation_connectedness,
    build_connectedness_summary,
    create_connectedness_edges,
    describe_gfevd_requirements,
)

SENTIMENT_OUT = OUTPUTS_DIR / "sentiment"
MERGED_OUT = OUTPUTS_DIR / "merged"
CONN_OUT = OUTPUTS_DIR / "connectedness"
for directory in [SENTIMENT_OUT, MERGED_OUT, CONN_OUT, FIGURES_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

## Raw file availability check

In [2]:
expected_raw = ["merged_data.json", "scraped_news.csv", "sentiment_log.csv"]
raw_file_status = []
for name in expected_raw:
    path = RAW_DATA_DIR / name
    raw_file_status.append({
        "file_name": name,
        "path": str(path.relative_to(PROJECT_ROOT)),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
        "status": "found" if path.exists() else "missing",
    })
raw_file_status = pd.DataFrame(raw_file_status)
raw_file_status

,file_name,path,exists,size_bytes,status
0,merged_data.json,data/raw/merged_data.json,True,451,found
1,scraped_news.csv,data/raw/scraped_news.csv,True,1,found
2,sentiment_log.csv,data/raw/sentiment_log.csv,True,1170,found


## Legacy script check

In [3]:
legacy_rows = []
for path in sorted(LEGACY_DIR.glob("*.py")):
    text = path.read_text(encoding="utf-8", errors="ignore")
    placeholder = "original prototype file not found" in text.lower() or "placeholder" in text.lower()
    legacy_rows.append({
        "script": path.name,
        "line_count": len(text.splitlines()),
        "status": "placeholder" if placeholder else "available",
        "notes": "Needs original script" if placeholder else "Recovered script available",
    })
legacy_status = pd.DataFrame(legacy_rows)
legacy_status

,script,line_count,status,notes
0,step1_sentiment_test.py,32,available,Recovered script available
1,step2_scraper.py,45,available,Recovered script available
2,step3_stock_filter.py,53,available,Recovered script available
3,step4_keyword_generator.py,63,available,Recovered script available
4,step5_dynamic_filtering.py,128,available,Recovered script available
5,step6_daily_sentiment.py,59,available,Recovered script available
6,step7_merge_data.py,43,available,Recovered script available
7,step8_gfevd_analysis.py,60,available,Recovered script available
8,step9_visualize.py,17,available,Recovered script available


## Load scraped news

In [4]:
scraped_path = RAW_DATA_DIR / "scraped_news.csv"
try:
    scraped_news = load_csv(scraped_path) if scraped_path.exists() else pd.DataFrame()
except Exception as exc:
    scraped_news = pd.DataFrame()
    print(f"scraped_news.csv could not be parsed: {exc}")

standardized_news = standardize_scraped_news(scraped_news) if not scraped_news.empty else pd.DataFrame()
print(f"Scraped news rows: {len(standardized_news)}")
standardized_news.head()

Scraped news rows: 0


""


## Load sentiment log

In [5]:
sentiment_path = RAW_DATA_DIR / "sentiment_log.csv"
sentiment_log = load_csv(sentiment_path) if sentiment_path.exists() else pd.DataFrame()
standardized_sentiment = standardize_sentiment_log(sentiment_log) if not sentiment_log.empty else pd.DataFrame()

# Optional OpenAI scoring: uses local .env if available and falls back silently to rules.
# The API key is never printed or saved.
if not standardized_sentiment.empty and "headline" in standardized_sentiment.columns:
    standardized_sentiment = score_sentiment_dataframe(
        standardized_sentiment,
        text_col="headline",
        use_openai=True,
    )

if not standardized_sentiment.empty:
    save_dataframe(standardized_sentiment, SENTIMENT_OUT / "standardized_sentiment_log.csv")
    if "scoring_method" in standardized_sentiment.columns and (standardized_sentiment["scoring_method"] == "openai_structured").any():
        openai_cols = [
            col for col in [
                "date", "company", "ticker", "headline", "sentiment_score", "sentiment_label",
                "confidence", "rationale", "risk_flags", "recommended_signal", "scoring_method"
            ] if col in standardized_sentiment.columns
        ]
        save_dataframe(standardized_sentiment[openai_cols], SENTIMENT_OUT / "openai_sentiment_scores.csv")

print(f"Sentiment rows: {len(standardized_sentiment)}")
if "scoring_method" in standardized_sentiment.columns:
    print(standardized_sentiment["scoring_method"].value_counts().to_string())
standardized_sentiment.head()

Sentiment rows: 11
scoring_method
openai_structured    11


,date,company,headline,sentiment_score,action,sentiment_label,recommended_signal,rescored_company,ticker,rescored_sentiment_score,rescored_sentiment_label,confidence,rationale,risk_flags,rescored_recommended_signal,scoring_method
0,2025-06-06,Tesla,Elon Musk gets more time to respond to US SEC ...,-0.20,HOLD,negative,HOLD,Tesla,TSLA,-0.20,negative,0.85,The news of Elon Musk being involved in a laws...,"Legal Issues, Regulatory Concerns",HOLD,openai_structured
1,2025-06-06,Apple,Circle stock soars after IPO as stablecoin gia...,0.75,HOLD,positive,BUY,Apple,AAPL,0.75,positive,0.85,The news highlights the positive sentiment sur...,"market volatility, IPO performance concerns",BUY,openai_structured
2,2025-06-06,Apple,Fed's Schmid: uncomfortable with looking throu...,-0.60,SELL,negative,SELL,Apple,AAPL,-0.60,negative,0.85,The statement by Fed's Schmid indicating disco...,"market volatility, policy changes, supply chai...",SELL,openai_structured
3,2025-06-06,Apple,Delta warns new tariffs could force it to stop...,-0.60,SELL,negative,SELL,Apple,AAPL,-0.60,negative,0.85,The warning about tariffs negatively impacts D...,"trade tariffs, supply chain disruption",SELL,openai_structured
4,2025-06-06,Apple,Remy Cointreau Pulls Long-Term Targets on Econ...,-0.40,SELL,negative,SELL,Apple,AAPL,-0.40,negative,0.85,The news indicates uncertainty regarding econo...,"Economic Uncertainty, Tariff Risk",SELL,openai_structured


## Load merged data

In [6]:
merged_path = RAW_DATA_DIR / "merged_data.json"
merged_raw = load_json_records(merged_path) if merged_path.exists() else pd.DataFrame()
merged_market = standardize_market_columns(merged_raw) if not merged_raw.empty else pd.DataFrame()
merged_market = calculate_market_returns(merged_market) if not merged_market.empty else pd.DataFrame()

print(f"Merged raw rows: {len(merged_market)}")
merged_market.head()

Merged raw rows: 2


,date,sentiment_score,company,headline,action,date_2,close_amzn,high_amzn,low_amzn,open_amzn,volume_amzn,close_amzn_return
0,2025-06-06,-0.316967,"Apple, Tesla",6,SELL,2025-06-06,213.570007,213.869995,210.5,212.399994,39832500.0,NaN
1,2025-10-20,-0.255540,Amazon,5,HOLD,NaT,NaN,NaN,NaN,NaN,NaN,NaN


## Standardize sentiment fields

In [7]:
sentiment_by_date = summarize_sentiment_by_date(standardized_sentiment) if not standardized_sentiment.empty else pd.DataFrame()
sentiment_by_company = summarize_sentiment_by_company(standardized_sentiment) if not standardized_sentiment.empty else pd.DataFrame()
signal_summary = create_signal_summary(standardized_sentiment) if not standardized_sentiment.empty else pd.DataFrame()

if not sentiment_by_date.empty:
    save_dataframe(sentiment_by_date, SENTIMENT_OUT / "sentiment_by_date.csv")
if not sentiment_by_company.empty:
    save_dataframe(sentiment_by_company, SENTIMENT_OUT / "sentiment_by_company.csv")
if not signal_summary.empty:
    save_dataframe(signal_summary, SENTIMENT_OUT / "signal_summary.csv")

display(sentiment_by_date)
display(sentiment_by_company)
display(signal_summary)

,date,record_count,avg_sentiment_score,negative_count,neutral_count,positive_count
0,2025-06-06,6,-0.141667,4,0,2
1,2025-10-20,5,-0.340000,4,1,0


,company_key,record_count
0,Amazon,5
1,Apple,5
2,Tesla,1


,action,record_count,share
0,SELL,7,0.636364
1,HOLD,3,0.272727
2,BUY,1,0.090909


## Standardize market fields

In [8]:
market_summary = summarize_market_data(merged_market) if not merged_market.empty else {"status": "no market data"}
market_columns = identify_market_columns(merged_market) if not merged_market.empty else {}
print(market_summary)
print(market_columns)

{'rows': 2, 'columns': 12, 'market_column_groups': {'date_columns': ['date', 'sentiment_score', 'date_2'], 'ticker_columns': [], 'price_columns': ['close_amzn', 'high_amzn', 'low_amzn', 'open_amzn', 'close_amzn_return'], 'return_columns': ['close_amzn_return'], 'volume_columns': ['volume_amzn']}, 'numeric_columns': ['sentiment_score', 'headline', 'close_amzn', 'high_amzn', 'low_amzn', 'open_amzn', 'volume_amzn', 'close_amzn_return'], 'numeric_missing_total': 7}
{'date_columns': ['date', 'sentiment_score', 'date_2'], 'ticker_columns': [], 'price_columns': ['close_amzn', 'high_amzn', 'low_amzn', 'open_amzn', 'close_amzn_return'], 'return_columns': ['close_amzn_return'], 'volume_columns': ['volume_amzn']}


## Merge sentiment and market data

In [9]:
if not standardized_sentiment.empty and not merged_market.empty:
    clean_merged = merge_sentiment_market(standardized_sentiment, merged_market)
elif not merged_market.empty:
    clean_merged = merged_market.copy()
elif not standardized_sentiment.empty:
    clean_merged = standardized_sentiment.copy()
else:
    clean_merged = pd.DataFrame()

if not clean_merged.empty:
    save_dataframe(clean_merged, MERGED_OUT / "clean_merged_sentiment_market.csv")

print(inspect_merged_dataset(clean_merged) if not clean_merged.empty else "No merged dataset available")
print(identify_join_keys(clean_merged) if not clean_merged.empty else "No join keys available")
clean_merged.head()

{'rows': 2, 'columns': 16, 'column_names': ['date', 'avg_sentiment_score', 'news_count', 'companies', 'dominant_action', 'sentiment_score', 'company', 'headline', 'action', 'date_2', 'close_amzn', 'high_amzn', 'low_amzn', 'open_amzn', 'volume_amzn', 'close_amzn_return'], 'missing_values_total': 8, 'duplicate_rows': 0}
{'date_keys': ['date', 'avg_sentiment_score', 'sentiment_score', 'date_2'], 'company_keys': ['company'], 'article_keys': ['headline']}


,date,avg_sentiment_score,news_count,companies,dominant_action,sentiment_score,company,headline,action,date_2,close_amzn,high_amzn,low_amzn,open_amzn,volume_amzn,close_amzn_return
0,2025-06-06,-0.141667,6,"Apple, Tesla",SELL,-0.316967,"Apple, Tesla",6,SELL,2025-06-06,213.570007,213.869995,210.5,212.399994,39832500.0,NaN
1,2025-10-20,-0.340000,5,Amazon,HOLD,-0.255540,Amazon,5,HOLD,NaT,NaN,NaN,NaN,NaN,NaN,NaN


## Signal summary

The `BUY`, `HOLD`, and `SELL` labels are simple sentiment-derived research labels. Phase 1B uses `SELL` for scores at or below `-0.25`, `BUY` for scores at or above `0.25`, and `HOLD` otherwise. Existing `action` values are preserved separately when present. These labels are not trading advice or a trading recommendation system.

In [10]:
signal_summary

,action,record_count,share
0,SELL,7,0.636364
1,HOLD,3,0.272727
2,BUY,1,0.090909


## Market returns

In [11]:
return_cols = [col for col in clean_merged.columns if str(col).endswith("_return")] if not clean_merged.empty else []
print("Return columns:", return_cols)
clean_merged[["date"] + return_cols].head() if return_cols and "date" in clean_merged.columns else pd.DataFrame()

Return columns: ['close_amzn_return']


,date,close_amzn_return
0,2025-06-06,NaN
1,2025-10-20,NaN


## Connectedness input preparation

In [12]:
modeling_dataset = create_modeling_dataset(clean_merged) if not clean_merged.empty else pd.DataFrame()
connectedness_inputs = prepare_connectedness_inputs(modeling_dataset) if not modeling_dataset.empty else pd.DataFrame()

print(describe_gfevd_requirements())
print("Connectedness input shape:", connectedness_inputs.shape)
connectedness_inputs.head()

{'required_shape': 'Time-indexed panel with at least two numeric market or sentiment series.', 'required_cleaning': 'Consistent dates, no duplicated date-series keys, and handled missing values.', 'required_validation': 'Stationarity or transformation review, lag-order choice, and sensitivity checks.', 'phase_0_status': 'Inspection only. Full GFEVD repair is planned for Phase 1.'}
Connectedness input shape: (2, 4)


,avg_sentiment_score,news_count,sentiment_score,headline
date,,,,
2025-06-06,-0.141667,6,-0.316967,6
2025-10-20,-0.340000,5,-0.255540,5


## Correlation or GFEVD-style connectedness analysis

Formal GFEVD requires a longer, clean time series for VAR estimation. The recovered dataset remains too small, so the notebook uses an absolute-correlation connectedness fallback and documents the limitation.

In [13]:
if connectedness_inputs.shape[0] >= 2 and connectedness_inputs.shape[1] >= 2:
    connectedness_matrix = calculate_correlation_connectedness(connectedness_inputs)
else:
    connectedness_matrix = pd.DataFrame()

connectedness_summary = build_connectedness_summary(connectedness_inputs)
connectedness_edges = create_connectedness_edges(connectedness_matrix, threshold=0.2)

connectedness_matrix.to_csv(CONN_OUT / "connectedness_matrix.csv")
save_dataframe(connectedness_edges, CONN_OUT / "connectedness_edges.csv")
save_dataframe(connectedness_summary, CONN_OUT / "connectedness_summary.csv")

display(connectedness_matrix)
display(connectedness_edges)
display(connectedness_summary)

,avg_sentiment_score,news_count,sentiment_score,headline
avg_sentiment_score,1.0,1.0,1.0,1.0
news_count,1.0,1.0,1.0,1.0
sentiment_score,1.0,1.0,1.0,1.0
headline,1.0,1.0,1.0,1.0


,source,target,connectedness
1,avg_sentiment_score,sentiment_score,1.0
4,news_count,sentiment_score,1.0
6,sentiment_score,avg_sentiment_score,1.0
7,sentiment_score,news_count,1.0
0,avg_sentiment_score,news_count,1.0
2,avg_sentiment_score,headline,1.0
3,news_count,avg_sentiment_score,1.0
5,news_count,headline,1.0
9,headline,avg_sentiment_score,1.0
10,headline,news_count,1.0


,metric,value
0,method,absolute_correlation_fallback
1,series_count,4
2,observation_count,2
3,average_off_diagonal_connectedness,1.0
4,max_pairwise_connectedness,1.0


## Figures

In [14]:
created_figures = []

if not standardized_sentiment.empty and "sentiment_score" in standardized_sentiment.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    standardized_sentiment["sentiment_score"].dropna().hist(ax=ax, bins=10, color="#2563eb")
    ax.set_title("Sentiment Score Distribution")
    ax.set_xlabel("Sentiment score")
    ax.set_ylabel("Record count")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "sentiment_distribution.png", dpi=160, bbox_inches="tight")
    plt.close(fig)
    created_figures.append("reports/figures/sentiment_distribution.png")

if not sentiment_by_company.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    sentiment_by_company.head(10).sort_values("record_count").plot(kind="barh", x="company_key", y="record_count", ax=ax, color="#059669", legend=False)
    ax.set_title("Sentiment Records by Company")
    ax.set_xlabel("Record count")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "sentiment_by_company.png", dpi=160, bbox_inches="tight")
    plt.close(fig)
    created_figures.append("reports/figures/sentiment_by_company.png")

if not sentiment_by_date.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(sentiment_by_date["date"], sentiment_by_date["avg_sentiment_score"], marker="o", color="#7c3aed")
    ax.set_title("Average Sentiment Over Time")
    ax.set_xlabel("Date")
    ax.set_ylabel("Average sentiment score")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "sentiment_over_time.png", dpi=160, bbox_inches="tight")
    plt.close(fig)
    created_figures.append("reports/figures/sentiment_over_time.png")

if not signal_summary.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    signal_summary.plot(kind="bar", x="action", y="record_count", ax=ax, color="#dc2626", legend=False)
    ax.set_title("Simple Signal Label Summary")
    ax.set_xlabel("Signal label")
    ax.set_ylabel("Record count")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "signal_summary.png", dpi=160, bbox_inches="tight")
    plt.close(fig)
    created_figures.append("reports/figures/signal_summary.png")

if not connectedness_matrix.empty:
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(connectedness_matrix.values, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(connectedness_matrix.columns)))
    ax.set_xticklabels(connectedness_matrix.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(connectedness_matrix.index)))
    ax.set_yticklabels(connectedness_matrix.index)
    ax.set_title("Connectedness Heatmap")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "connectedness_heatmap.png", dpi=160, bbox_inches="tight")
    plt.close(fig)
    created_figures.append("reports/figures/connectedness_heatmap.png")

created_figures

['reports/figures/sentiment_distribution.png',
 'reports/figures/sentiment_by_company.png',
 'reports/figures/sentiment_over_time.png',
 'reports/figures/signal_summary.png',
 'reports/figures/connectedness_heatmap.png']

## Business interpretation

The recovered sentiment log is usable for demonstrating a safer research pipeline, but it remains small. The pipeline can now optionally use structured OpenAI scoring from a local `.env` key, while falling back to rule-based scoring without exposing sensitive values. The merged market file includes AMZN price fields but only two daily records, with one missing market observation. Because of this, connectedness output remains an exploratory correlation fallback, not a formal GFEVD result.

## Limitations

- `scraped_news.csv` exists but is empty.
- The sentiment log has a small number of records.
- OpenAI scoring is optional and falls back to transparent rules if unavailable.
- The merged market file has only two rows and missing market values.
- Formal GFEVD is not statistically valid from this small dataset.
- Signal labels are not trading advice.
- No heavy scraping was run.

## Next steps for Phase 2 polish

- Polish README and reports.
- Keep the connectedness fallback limitation visible.
- Add career-facing outputs.
- Avoid overclaiming trading or forecasting performance.